# 05 — Stationarity, Mean Reversion and Half-Life

**Purpose:** Characterize residual dynamics per pair: is there statistically credible, economically exploitable mean reversion (mandate §4.1/§4.2)?

**Research questions:**
1. Is each pair's residual stationary (ADF+KPSS combined verdict), per year and full-sample?
2. What is the half-life distribution (AR(1) and OU), and is it inside the tradable band (research_config: min/max_half_life_bars)?
3. Do results survive the roll-exclusion mask from notebook 01?
4. How much of measured half-life is hedge-noise artifact (L-007)?

**Data used:** residuals from notebook 04's selected hedges (**blocked**, L-001).


In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import yaml

RESEARCH_CONFIG = yaml.safe_load(open("../config/research_config.yaml"))
SEED = RESEARCH_CONFIG["meta"]["random_seed"]
np.random.seed(SEED)
print(f"config loaded | global seed = {SEED}")


## Methodology

`stationarity.stationarity_verdict` + `rolling_adf_pvalue` (yearly windows); `mean_reversion.half_life_ar1` and `fit_ou` full-sample and per-year; `mean_reversion.convergence_study` at entry thresholds 1.5/2.0/2.5/3.0 → convergence rate, time-to-converge, max adverse |z|; all repeated with roll exclusions on/off. L-007 control: recompute half-life with hedge refit frequency varied — divergence measures artifact share.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/processed")
DATA_AVAILABLE = any(DATA_DIR.glob("*_minute.*")) if DATA_DIR.exists() else False
if not DATA_AVAILABLE:
    print("BLOCKED-ON-DATA: no futures market data in this environment (see "
          "reports/00_repository_audit.md, issue L-001).\n"
          "Run this notebook inside QuantConnect Research, or drop licensed data\n"
          "into data/processed/ in the canonical schema (src/spread_research/data_loader.py).")


In [ ]:
if DATA_AVAILABLE:
    from spread_research.mean_reversion import half_life_ar1, fit_ou, convergence_study
    from spread_research.stationarity import stationarity_verdict
    print("wire per-pair residual characterization here")

## Results

**BLOCKED-ON-DATA** — this section intentionally contains no results. No synthetic or fabricated market findings are presented as evidence (CLAUDE.md gate 3). It will be populated when the notebook runs against real data.

## Limitations

Half-life estimates on autocorrelated residuals have wide confidence bands; per-year dispersion is reported, never just a point estimate.

## Decision

Pairs leave the funnel here if: verdict non-stationary/ambiguous in ≥2 years, or half-life outside the tradable band, or convergence rate at z=2 below 60% within max_holding_bars.

## What this means for the algorithm

Surviving pairs' half-lives set exit-timeout and z-lookback scales; convergence statistics set realistic expectations for holding periods and stop distances.